# 03 — Modelling walkthrough (M3 features → M5 classical → M6 GRU/LSTM)

In [ ]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
from src.utils import load_config, set_seed
config = load_config()
set_seed(config["seed"])
RAW_DIR = PROJECT_ROOT / "data" / "raw"

In [ ]:
clean = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'flowcast_clean.parquet')

## M3 — features for horizon h = 1 (FR-05)
Expected: 172,800 rows (first 7 days dropped as warm-up), no NULLs in feature columns.

In [ ]:
from src.features import build_features, get_feature_columns, check_no_nulls
table = build_features(clean, config, horizon=1)
feature_columns = get_feature_columns(table)
check_no_nulls(table, feature_columns)
print(table.shape, len(feature_columns))
print(table['split'].value_counts())

## Leakage check by hand
Every traffic feature ends in `_t0`/`_t1`/`_t2` (origin = T − h) or is a seasonal lag. Nothing from window T itself:

In [ ]:
same_window = [c for c in feature_columns if c in ['traffic_volume', 'vehicle_count', 'avg_speed', 'occupancy', 'travel_time']]
print('same-window columns in features:', same_window)
assert same_window == []

## Naive baseline first
Persistence: ŷ(T) = volume at t0.

In [ ]:
from src.evaluate import regression_metrics
test = table.loc[(table['split'] == 'test') & table['mask_volume']]
regression_metrics(test['y_volume'], test['vol_t0'])

## Linear regression from scratch (build-it-once rule)
Cost J(w,b) = (1/2m) Σ (Xw + b − y)²; gradient ∂J/∂w = (1/m) Xᵀ(Xw + b − y). Expected: cost falls every epoch and the result matches sklearn closely.

In [ ]:
from src.ml_models import LinearRegressionGD, select_rows
from sklearn.linear_model import LinearRegression
train = select_rows(table, 'train', 'mask_volume'); val = select_rows(table, 'val', 'mask_volume')
X_train, y_train = train[feature_columns].to_numpy(float), train['y_volume'].to_numpy(float)
X_val, y_val = val[feature_columns].to_numpy(float), val['y_volume'].to_numpy(float)
gd = LinearRegressionGD(learning_rate=0.1, epochs=600).fit(X_train, y_train)
print('first / last cost:', gd.cost_history[0], gd.cost_history[-1])
print('GD val:', regression_metrics(y_val, gd.predict(X_val)))
print('sklearn val:', regression_metrics(y_val, LinearRegression().fit(X_train, y_train).predict(X_val)))

## Full M5 + M6 (long — 20-40 min on a laptop CPU)
Or run `python -m src.pipeline` from a terminal.

In [ ]:
# from src.features import build_all_horizons
# from src.ml_models import run_classical
# from src.dl_model import run_deep_learning
# tables = build_all_horizons(clean, config)
# runs, scoreboard, registry = run_classical(tables, config)
# run_deep_learning(clean, tables[1], config)

## Scoreboard (one table, every model × target × split)

In [ ]:
board = pd.read_csv(PROJECT_ROOT / 'reports' / 'scoreboard.csv')
board.loc[(board['split'] == 'test') & (board['horizon'] == 1), ['model', 'target', 'family', 'rmse', 'mape', 'macro_f1', 'roc_auc', 'pr_auc']].round(4)